<a href="https://colab.research.google.com/github/ibelalov/rlmw/blob/main/rlmw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# rlmw

Single-notebook development environment for low-weight span-vector search.

Core components:

1. Binary linear algebra over F_2
2. Planted span instance generator
3. Bitset/popcount utilities
4. Direction bank construction
5. Exact discrete-gradient descent
6. Local-minimum support-intersection constraints
7. Failed-local-minimum archive
8. Neural direction model
9. Cross-entropy sampler
10. Strategy-level Q-controller
11. Exact local-minimum intersection solver
12. Training/evaluation dashboard

In [2]:
# @title
# --- 00. Environment setup ---

import os
import sys
import math
import time
import json
import random
import pathlib
import itertools
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("PyTorch available:", TORCH_AVAILABLE)
if TORCH_AVAILABLE:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.0.2
PyTorch available: True
PyTorch: 2.10.0+cpu
CUDA available: False


In [7]:
# @title Environment detection and storage backend selection
import os


def _env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip().lower() in {"1", "true", "yes", "on"}


# Small self-tests for environment-flag parsing
assert _env_flag("__RLMW_TEST_UNSET__", "0") is False
assert _env_flag("__RLMW_TEST_UNSET__", "1") is True

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

HEADLESS = _env_flag("RLMW_HEADLESS", "0")
SMOKE = _env_flag("RLMW_SMOKE", "0")
USE_DRIVE = IN_COLAB and (not HEADLESS)

if USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [9]:
# @title Project paths
import os
from pathlib import Path

if USE_DRIVE:
    PROJECT_ROOT = Path("/content/drive/MyDrive") / "rlmw"
else:
    PROJECT_ROOT = Path(os.environ.get("RLMW_PROJECT_ROOT", "/tmp/rlmw"))

DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
EXPORT_DIR = PROJECT_ROOT / "exports"
CACHE_DIR = PROJECT_ROOT / "cache"

for p in [PROJECT_ROOT, DATA_DIR, RUNS_DIR, CHECKPOINT_DIR, EXPORT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("HEADLESS:", HEADLESS)
print("SMOKE:", SMOKE)
print("USE_DRIVE:", USE_DRIVE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("CACHE_DIR:", CACHE_DIR)

Project root: /content/drive/MyDrive/rlmw
Data dir: /content/drive/MyDrive/rlmw/data
Runs dir: /content/drive/MyDrive/rlmw/runs
Checkpoint dir: /content/drive/MyDrive/rlmw/checkpoints
Export dir: /content/drive/MyDrive/rlmw/exports
Cache dir: /content/drive/MyDrive/rlmw/cache
